In [2]:
import warnings
warnings.filterwarnings("ignore")


In [1]:
import os
from bs4 import BeautifulSoup
from selenium.common.exceptions import TimeoutException, WebDriverException,NoSuchElementException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time,random
import pandas as pd
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import logging

from time import sleep
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
os.environ['PASSWORD']

'ezar@@25'

In [3]:


# Setup ChromeOptions
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

# Setup ChromeDriver with Service

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.linkedin.com/login')
email=driver.find_element(By.ID,'username')
email.send_keys(os.environ['EMAIL'])
password=driver.find_element(By.ID,'password')
password.send_keys(os.environ['PASSWORD'])
password.submit()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [4]:


def navigate_to_search_results(driver, query="manufacturing and Tunisia", timeout=8, save_source_on_error=False):
    try:
        # Vérifier l'URL actuelle
        logger.info(f"Current URL: {driver.current_url}")
        if "feed" not in driver.current_url:
            logger.info("Navigating to LinkedIn homepage")
            driver.get("https://www.linkedin.com/feed/")
            WebDriverWait(driver, timeout).until(EC.url_contains("feed"))

        # Attendre que la page soit chargée
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        logger.info("Page loaded")

        # Gérer les popups
        try:
            dismiss_button = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((
                    By.XPATH, 
                    "//button[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'accept') or contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'dismiss') or contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'close') or contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'reject')]"
                ))
            )
            dismiss_button.click()
            logger.info("Dismissed popup")
            time.sleep(random.uniform(0.5, 1.0))  # Pause aléatoire pour imiter un humain
        except TimeoutException:
            logger.debug("No popup detected")

        # Cliquer sur le bouton de recherche
        try:
            search_button = WebDriverWait(driver, timeout // 2).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "search-global-typeahead__collapsed-search-button"))
            )
            search_button.click()
            logger.info("Clicked search button")
            time.sleep(random.uniform(0.5, 1.0))  # Pause aléatoire pour activer la barre
        except TimeoutException:
            logger.warning("Search button not found, attempting search bar directly")

        # Localiser la barre de recherche
        search_bar = WebDriverWait(driver, timeout // 2).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "search-global-typeahead__input"))
        )
        logger.info("Search bar found")

        # Vérifier si la barre de recherche est activée
        if not driver.execute_script("return !arguments[0].disabled;", search_bar):
            logger.error("Search bar is disabled")
            if save_source_on_error:
                with open("page_source_error.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                logger.info("Page source saved to page_source_error.html")
            return False

        # Effacer et envoyer la requête
        search_bar.clear()
        search_bar.send_keys(query)
        search_bar.send_keys(Keys.RETURN)
        logger.info(f"Search query sent: {query}")

        # Attendre les résultats
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((
                By.XPATH, 
                "//button[text()='Companies'] | //div[contains(@class, 'search-results-container')]"
            ))
        )
        logger.info("Search results loaded")
        return True

    except TimeoutException as e:
        logger.error(f"Timeout error: {e}")
        if save_source_on_error:
            with open("page_source_error.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            logger.info("Page source saved to page_source_error.html")
        return False
    except WebDriverException as e:
        logger.error(f"WebDriver error: {e}")
        if save_source_on_error:
            with open("page_source_error.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            logger.info("Page source saved to page_source_error.html")
        return False
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        if save_source_on_error:
            with open("page_source_error.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            logger.info("Page source saved to page_source_error.html")
        return False


In [16]:
navigate_to_search_results(driver, query="Transport AND Tunisia", timeout=10)

2025-08-22 10:41:28,110 - INFO - Current URL: https://www.linkedin.com/search/results/companies/?keywords=Retail%20management%20AND%20Tunisia&origin=SWITCH_SEARCH_VERTICAL&page=4&sid=o((


2025-08-22 10:41:28,121 - INFO - Navigating to LinkedIn homepage
2025-08-22 10:41:35,959 - INFO - Page loaded
2025-08-22 10:41:39,554 - INFO - Clicked search button
2025-08-22 10:41:40,468 - INFO - Search bar found
2025-08-22 10:41:40,784 - INFO - Search query sent: Transport AND Tunisia
2025-08-22 10:41:42,570 - INFO - Search results loaded


True

In [6]:
def navigate_to_companies_tab(driver, timeout=8, save_source_on_error=False):
    """
    Navigue vers l'onglet Entreprises dans les résultats de recherche LinkedIn.
    """
    try:
        logger.info("Step 7: Navigating to Entreprises tab")
        # Vérifier que la page des résultats est chargée
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'search-results-container')]"))
        )
        logger.info(f"Search results page loaded. Current URL: {driver.current_url}")

        # Vérifier si l'onglet Entreprises est déjà actif via l'URL
        if "/search/results/companies/" in driver.current_url:
            logger.info("Entreprises tab already active, skipping click")
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((
                    By.XPATH, 
                    "//div[contains(@class, 'search-results-container')]//ul[contains(@class, 'search-results__list')]"
                ))
            )
            logger.info("Entreprises tab content loaded")
            return True

        # Vérifier la présence des boutons de filtre
        filter_buttons = driver.find_elements(By.XPATH, "//button[contains(@class, 'search-reusables__filter-pill-button')]")
        if not filter_buttons:
            logger.error("No filter buttons found with class 'search-reusables__filter-pill-button'")
            if save_source_on_error:
                with open("page_source_step7.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                logger.info("Page source saved to page_source_step7.html")
            return False

        # Nouvelle approche : Utiliser un XPATH basé sur data-control-name ou aria-label
        companies_tab = None
        try:
            companies_tab = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((
                    By.XPATH, 
                    "//button[contains(@class, 'search-reusables__filter-pill-button') and (contains(@data-control-name, 'filter_companies') or contains(@aria-label, 'Entreprises') or contains(@aria-label, 'Companies'))]"
                ))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", companies_tab)
            time.sleep(random.uniform(0.3, 0.7))
            driver.execute_script("arguments[0].click();", companies_tab)
            logger.info("Clicked Entreprises tab")
        except TimeoutException:
            logger.warning("Button with data-control-name or aria-label not found, attempting text-based fallback")
            try:
                companies_tab = WebDriverWait(driver, timeout // 2).until(
                    EC.element_to_be_clickable((
                        By.XPATH, 
                        "//button[contains(@class, 'search-reusables__filter-pill-button') and (contains(., 'Entreprises') or contains(., 'Companies'))]"
                    ))
                )
                driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", companies_tab)
                time.sleep(random.uniform(0.3, 0.7))
                driver.execute_script("arguments[0].click();", companies_tab)
                logger.info("Clicked Entreprises tab (text-based fallback)")
            except TimeoutException:
                logger.error("No button with text 'Entreprises' or 'Companies' found")
                if save_source_on_error:
                    with open("page_source_step7.html", "w", encoding="utf-8") as f:
                        f.write(driver.page_source)
                    logger.info("Page source saved to page_source_step7.html")
                return False

        # Vérifier le chargement de l'onglet via l'URL ou le conteneur
        WebDriverWait(driver, timeout).until(
            lambda d: "/search/results/companies/" in d.current_url or
                      EC.presence_of_element_located((
                          By.XPATH, 
                          "//div[contains(@class, 'search-results-container')]//ul[contains(@class, 'search-results__list')]"
                      ))(d)
        )
        logger.info("Entreprises tab content loaded")
        return True

    except (TimeoutException, WebDriverException) as e:
        logger.error(f"Error in Step 7 (navigating to Entreprises tab): {e}")
        if save_source_on_error:
            with open("page_source_step7.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            logger.info("Page source saved to page_source_step7.html")
        return False

In [17]:
navigate_to_companies_tab(driver, timeout=6, save_source_on_error=False)


2025-08-22 10:41:50,917 - INFO - Step 7: Navigating to Entreprises tab
2025-08-22 10:41:50,937 - INFO - Search results page loaded. Current URL: https://www.linkedin.com/search/results/all/?keywords=Transport%20AND%20Tunisia&origin=GLOBAL_SEARCH_HEADER&sid=QTa
2025-08-22 10:41:57,191 - WARNING - Button with data-control-name or aria-label not found, attempting text-based fallback
2025-08-22 10:41:57,603 - INFO - Clicked Entreprises tab (text-based fallback)
2025-08-22 10:41:58,309 - INFO - Entreprises tab content loaded


True

In [55]:
driver.quit()

In [14]:
def collect_company_links(driver, max_companies=60, timeout=6, save_source_on_error=False):
    """
    Collecte les URLs des max_companies premières entreprises dans les résultats.
    
    """
    
    unique_hrefs = set()
    try:
        logger.info(f"Step 8: Collecting links to first {max_companies} companies")
        # Vérifier que la page des résultats est chargée
        WebDriverWait(driver, timeout).until(
            lambda d: "/search/results/companies/" in d.current_url or
                      EC.presence_of_element_located((
                          By.XPATH, 
                          "//div[contains(@class, 'search-results-container')]"
                      ))(d)
        )
        logger.info("Search results container loaded")

        # Vérifier si l'onglet Entreprises est actif
        try:
            companies_tab = WebDriverWait(driver, timeout // 2).until(
                EC.presence_of_element_located((
                    By.XPATH, 
                    "//button[contains(@class, 'search-reusables__filter-pill-button') and contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'entreprises') and contains(@class, 'artdeco-pill--selected')]"
                ))
            )
            logger.info("Confirmed Entreprises tab is active")
        except TimeoutException:
            logger.warning("Entreprises tab not active, attempting to click again")
            try:
                companies_tab = WebDriverWait(driver, timeout // 2).until(
                    EC.element_to_be_clickable((
                        By.XPATH, 
                        "//button[contains(@class, 'search-reusables__filter-pill-button') and contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'entreprises')]"
                    ))
                )
                driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", companies_tab)
                time.sleep(random.uniform(0.2, 0.4))
                driver.execute_script("arguments[0].click();", companies_tab)
                logger.info("Clicked Entreprises tab")
                WebDriverWait(driver, timeout).until(
                    EC.presence_of_element_located((
                        By.XPATH, 
                        "//div[contains(@class, 'search-results-container')]"
                    ))
                )
            except TimeoutException:
                logger.warning("Entreprises tab not found, proceeding with current results")

        # Collecter les liens sur plusieurs pages si nécessaire
        page_number = 1
        while len(unique_hrefs) < max_companies:
            logger.info(f"Processing page {page_number} of search results")
            # Défilement progressif pour charger les résultats
            scroll_attempts = 3
            company_elements = []
            for attempt in range(scroll_attempts):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight * arguments[0]);", (attempt + 1) / scroll_attempts)
                time.sleep(random.uniform(0.2, 0.4))
                logger.info(f"Scrolled to {((attempt + 1) / scroll_attempts) * 100}% of page height")

                # Collecter les éléments des entreprises
                try:
                    elements = WebDriverWait(driver, timeout).until(
                        EC.presence_of_all_elements_located((
                            By.XPATH, 
                            "//div[contains(@class, 'search-results-container')]//a[contains(@href, '/company/') and @data-test-app-aware-link]"
                        ))
                    )
                    logger.info(f"Found {len(elements)} potential company links after scroll attempt {attempt + 1}")
                    for element in elements:
                        href = element.get_attribute("href").split("?")[0]
                        if href not in unique_hrefs and len(unique_hrefs) < max_companies:
                            unique_hrefs.add(href)
                            company_elements.append(element)
                except TimeoutException:
                    logger.warning(f"No company links found after scroll attempt {attempt + 1}")
                    continue

            # Loguer les liens et noms pour débogage
            for i, element in enumerate(company_elements, len(unique_hrefs) - len(company_elements) + 1):
                try:
                    href = element.get_attribute("href").split("?")[0]
                    # Nouvelle approche pour les noms
                    try:
                        name_element = element.find_element(
                            By.XPATH, 
                            ".//span[contains(@class, 'entity-result__title')] | .//div[contains(@class, 'entity-result__title')] | .//span[contains(@class, 't-bold')] | .//h3 | .//h4 | .//span"
                        )
                        name = name_element.text.strip() or "Unknown"
                    except:
                        name = "Unknown"
                    logger.info(f"Collected company link {i}: {href} (Name: {name})")
                    company_links.append(href)
                except Exception as e:
                    logger.warning(f"Error processing company {i}: {e}")
                    continue

            if len(unique_hrefs) >= max_companies:
                break

            # Tenter la pagination
            try:
                next_button = WebDriverWait(driver, timeout // 2).until(
                    EC.element_to_be_clickable((
                        By.XPATH, 
                        "//button[contains(@aria-label, 'Suivant') or contains(@aria-label, 'Next') or contains(@class, 'artdeco-pagination__button--next')]"
                    ))
                )
                driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", next_button)
                time.sleep(random.uniform(0.2, 0.4))
                driver.execute_script("arguments[0].click();", next_button)
                logger.info(f"Clicked Next button for page {page_number + 1}")
                WebDriverWait(driver, timeout).until(
                    EC.presence_of_element_located((
                        By.XPATH, 
                        "//div[contains(@class, 'search-results-container')]"
                    ))
                )
                page_number += 1
            except TimeoutException:
                logger.warning(f"No Next button found on page {page_number}, stopping")
                break

        if len(company_links) < max_companies:
            logger.warning(f"Only collected {len(company_links)} company links out of {max_companies}")
        else:
            logger.info(f"Successfully collected {len(company_links)} company links")
        return company_links

    except (TimeoutException, WebDriverException) as e:
        logger.error(f"Error in Step 8 (collecting company links): {e}")
        if save_source_on_error:
            with open("page_source_step8.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            logger.info("Page source saved to page_source_step8.html")
        return company_links

In [18]:
company_links=collect_company_links(driver, max_companies=300, timeout=6, save_source_on_error=False)

2025-08-22 10:42:05,695 - INFO - Step 8: Collecting links to first 300 companies
2025-08-22 10:42:05,705 - INFO - Search results container loaded
2025-08-22 10:42:09,027 - WARNING - Entreprises tab not active, attempting to click again
2025-08-22 10:42:12,157 - WARNING - Entreprises tab not found, proceeding with current results
2025-08-22 10:42:12,157 - INFO - Processing page 1 of search results
2025-08-22 10:42:12,493 - INFO - Scrolled to 33.33333333333333% of page height
2025-08-22 10:42:12,507 - INFO - Found 20 potential company links after scroll attempt 1
2025-08-22 10:42:12,866 - INFO - Scrolled to 66.66666666666666% of page height
2025-08-22 10:42:12,878 - INFO - Found 20 potential company links after scroll attempt 2
2025-08-22 10:42:13,249 - INFO - Scrolled to 100.0% of page height
2025-08-22 10:42:13,259 - INFO - Found 20 potential company links after scroll attempt 3
2025-08-22 10:42:13,404 - INFO - Collected company link 1: https://www.linkedin.com/company/technology-&-str

In [11]:
print(len(company_links))

300


In [19]:
from selenium.common.exceptions import TimeoutException, WebDriverException,NoSuchElementException


In [20]:
import pandas as pd
df=pd.DataFrame(company_links,columns=['company_links'])
df.to_excel("company_links.xlsx",index=False)

In [87]:

company_data = []
try:
    logger.info("Extracting information from each company")
    for index, link in enumerate(company_links[:10], 1):
        logger.info(f"Processing company {index}/{len(company_links)}: {link}")
        data = {"Link": link, "Name": "N/A", "Description": "N/A", "Industry": "N/A", "Company Size": "N/A", "Headquarters": "N/A", "Website": "N/A"}

        try:
            # Navigate to company page
            try:
                driver.get(link)
                WebDriverWait(driver, 15).until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                )
                logger.info("Company page loaded")
            except TimeoutException:
                logger.error(f"Failed to load company page: {link}")
                company_data.append(data)
                continue

            # Check for CAPTCHA or login prompt
            try:
                captcha = driver.find_elements(By.XPATH, "//*[しろ*[@id='captcha']")
                login_prompt = driver.find_elements(By.ID, "username")
                if captcha or login_prompt:
                    logger.warning(f"CAPTCHA or login prompt detected on {link}. Saving page source.")
                    with open(f"page_source_company_{index}_captcha.html", "w", encoding="utf-8") as f:
                        f.write(driver.page_source)
                    company_data.append(data)
                    continue
            except Exception as e:
                logger.warning(f"Error checking for CAPTCHA/login: {e}")

            # Scroll to ensure content loads
            for _ in range(3):
                try:
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    logger.info("Scrolled to bottom of page")
                    time.sleep(random.uniform(1, 2))
                except Exception as e:
                    logger.warning(f"Scrolling failed: {e}")

            # Check if About tab is active
            try:
                active_tab = WebDriverWait(driver, 3).until(
                    EC.presence_of_element_located((
                        By.XPATH,
                        "//a[contains(@class, 'org-page-navigation__item-anchor') and @aria-current='page' and contains(@href, '/about/')]"
                    ))
                )
                logger.info("About tab is already active")
            except TimeoutException:
                # Navigate to About tab
                try:
                    about_tab_locators = [
                        (By.XPATH, "//a[contains(@class, 'org-page-navigation__item-anchor') and contains(@href, '/about/') and contains(text(), 'À propos')]"),
                        (By.XPATH, "//a[contains(@class, 'ember-view') and contains(@class, 'org-page-navigation__item-anchor') and contains(@href, '/about/')]"),
                        (By.XPATH, "//a[contains(text(), 'À propos') or contains(text(), 'About')]"),
                        (By.XPATH, "//a[@data-test-id='about']")
                    ]
                    about_tab = None
                    for by, value in about_tab_locators:
                        try:
                            about_tab = WebDriverWait(driver, 10).until(
                                EC.element_to_be_clickable((by, value))
                            )
                            logger.info(f"About tab found with {by}: {value}")
                            driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", about_tab)
                            driver.execute_script("arguments[0].click();", about_tab)
                            logger.info("Clicked About tab")
                            WebDriverWait(driver, 10).until(
                                EC.presence_of_element_located((
                                    By.XPATH,
                                    "//p[contains(@class, 'break-words') and contains(@class, 'white-space-pre-wrap')]"
                                ))
                            )
                            logger.info("About tab content loaded")
                            break
                        except TimeoutException:
                            logger.info(f"About tab locator {by}: {value} not found, trying next...")
                    if about_tab is None:
                        logger.warning("About tab not found, proceeding with current page")
                        nav_links = driver.find_elements(By.XPATH, "//a[contains(@class, 'org-page-navigation__item-anchor')]")
                        for link_elem in nav_links:
                            logger.info(f"Nav link found: text='{link_elem.text.strip()}', href='{link_elem.get_attribute('href') or 'None'}'")
                except Exception as e:
                    logger.warning(f"Error clicking About tab: {e}")

            # Extract data with retries
            max_retries = 2

            # Company Name
            for attempt in range(max_retries):
                try:
                    name = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((
                            By.XPATH,
                            "//h1[contains(@class, 'org-top-card-summary__title') or contains(@class, 'top-card__title') or @data-test-id='org-name' or contains(@class, 'org-about-us__name')]"
                        ))
                    ).text.strip()
                    data["Name"] = name
                    break
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Name"] = "N/A"
                        logger.warning("Company name not found after retries")
                    time.sleep(1)

            # Description
            for attempt in range(max_retries):
                try:
                    description = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((
                            By.XPATH,
                            "//p[contains(@class, 'break-words') and contains(@class, 'white-space-pre-wrap') and contains(@class, 't-black--light') and contains(@class, 'text-body-medium')]"
                        ))
                    ).text.strip()
                    data["Description"] = description
                    break
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Description"] = "N/A"
                        logger.warning("Description not found after retries")
                    time.sleep(1)

            # Industry (Sector)
            for attempt in range(max_retries):
                try:
                    # Try multiple strategies for finding industry
                    industry_selectors = [
                        # Strategy 1: Look for dd that follows dt containing "Industry" or "Secteur"
                        "//dt[contains(text(), 'Industry') or contains(text(), 'Secteur') or contains(text(), 'Industrie')]/following-sibling::dd[1]",
                        # Strategy 2: Direct class-based approach
                        "//dd[contains(@class, 't-black--light') and contains(@class, 'text-body-medium') and contains(@class, 'mb4')]",
                        # Strategy 3: Look for specific industry patterns
                        "//dd[contains(@class, 'mb4') and contains(@class, 't-black--light') and contains(@class, 'text-body-medium') and not(contains(text(), 'employés')) and not(contains(text(), 'employees'))]"
                    ]
                    
                    industry = None
                    for selector in industry_selectors:
                        try:
                            elements = driver.find_elements(By.XPATH, selector)
                            for element in elements:
                                text = element.text.strip()
                                # Filter out empty text and non-industry content
                                if text and not any(keyword in text.lower() for keyword in ['employés', 'employees', 'size', 'taille']):
                                    # Additional check: if it's not a location (doesn't contain common location indicators)
                                    if not any(location_indicator in text.lower() for location_indicator in [', ', 'texas', 'california', 'new york', 'paris', 'london']):
                                        industry = text
                                        break
                            if industry:
                                break
                        except:
                            continue
                    
                    if industry:
                        data["Industry"] = industry
                        logger.info(f"Industry found: {industry}")
                        break
                    else:
                        raise TimeoutException("Industry not found with any selector")
                        
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Industry"] = "N/A"
                        logger.warning("Industry not found after retries")
                    time.sleep(1)

            # Company Size
            for attempt in range(max_retries):
                try:
                    # Try multiple strategies for finding company size
                    size_selectors = [
                        # Strategy 1: Look for dd that follows dt containing size-related terms
                        "//dt[contains(text(), 'Company size') or contains(text(), 'Taille') or contains(text(), 'Size')]/following-sibling::dd[1]",
                        # Strategy 2: Look for dd containing employee-related text
                        "//dd[contains(@class, 't-black--light') and contains(@class, 'text-body-medium') and (contains(text(), 'employés') or contains(text(), 'employees'))]",
                        # Strategy 3: Look for dd with mb1 class (as shown in your example)
                        "//dd[contains(@class, 'mb1') and contains(@class, 't-black--light') and contains(@class, 'text-body-medium')]"
                    ]
                    
                    size = None
                    for selector in size_selectors:
                        try:
                            elements = driver.find_elements(By.XPATH, selector)
                            for element in elements:
                                text = element.text.strip()
                                # Look for employee/size indicators
                                if text and any(keyword in text.lower() for keyword in ['employés', 'employees', 'size', 'taille', '1-', '2-', '10', '50', '100', '500', '1000']):
                                    size = text
                                    break
                            if size:
                                break
                        except:
                            continue
                    
                    if size:
                        data["Company Size"] = size
                        logger.info(f"Company size found: {size}")
                        break
                    else:
                        raise TimeoutException("Company size not found with any selector")
                        
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Company Size"] = "N/A"
                        logger.warning("Company size not found after retries")
                    time.sleep(1)

             # Headquarters (Location)
            for attempt in range(max_retries):
                try:
                    # Try multiple strategies for finding headquarters
                    headquarters_selectors = [
                        # Strategy 1: Look for dd that follows dt containing headquarters-related terms
                        "//dt[contains(text(), 'Headquarters') or contains(text(), 'Siège') or contains(text(), 'Location')]/following-sibling::dd[1]",
                        # Strategy 2: Look for dd containing location patterns (City, State/Country)
                        "//dd[contains(@class, 't-black--light') and contains(@class, 'text-body-medium') and contains(@class, 'mb4') and contains(text(), ',')]",
                        # Strategy 3: Look for dd that contains common location patterns
                        "//dd[contains(@class, 'mb4') and contains(@class, 't-black--light') and contains(text(), ',') and not(contains(text(), 'employés')) and not(contains(text(), 'employees'))]"
                    ]
                    
                    headquarters = None
                    for selector in headquarters_selectors:
                        try:
                            elements = driver.find_elements(By.XPATH, selector)
                            for element in elements:
                                text = element.text.strip()
                                # Look for location patterns (should contain comma and not be employee count)
                                if text and ',' in text and not any(keyword in text.lower() for keyword in ['employés', 'employees', 'industrie', 'industry']):
                                    headquarters = text
                                    break
                            if headquarters:
                                break
                        except:
                            continue
                    
                    if headquarters:
                        data["Headquarters"] = headquarters
                        logger.info(f"Headquarters found: {headquarters}")
                        break
                    else:
                        raise TimeoutException("Headquarters not found with any selector")
                        
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Headquarters"] = "N/A"
                        logger.warning("Headquarters not found after retries")
                    time.sleep(1)

            # Add a fallback strategy to extract all dd elements and manually parse
            if data["Industry"] == "N/A" or data["Company Size"] == "N/A" or data["Headquarters"] == "N/A":
                logger.info("Attempting fallback extraction strategy")
                try:
                    # Get all dd elements with the common classes
                    all_dd_elements = driver.find_elements(By.XPATH, "//dd[contains(@class, 't-black--light') and contains(@class, 'text-body-medium')]")
                    
                    for element in all_dd_elements:
                        text = element.text.strip()
                        if not text:
                            continue
                            
                        # Try to categorize based on content
                        if data["Company Size"] == "N/A" and any(keyword in text.lower() for keyword in ['employés', 'employees', '1-', '2-', '10', '50', '100', '500', '1000']):
                            data["Company Size"] = text
                            logger.info(f"Fallback: Company size found: {text}")
                        elif data["Headquarters"] == "N/A" and ',' in text and not any(keyword in text.lower() for keyword in ['employés', 'employees', 'industrie', 'industry']):
                            data["Headquarters"] = text
                            logger.info(f"Fallback: Headquarters found: {text}")
                        elif data["Industry"] == "N/A" and not any(keyword in text.lower() for keyword in ['employés', 'employees', 'size', 'taille']) and ',' not in text:
                            # This might be industry if it doesn't contain employee info or location indicators
                            data["Industry"] = text
                            logger.info(f"Fallback: Industry found: {text}")
                            
                except Exception as e:
                    logger.warning(f"Fallback extraction failed: {e}")

            # Website
            for attempt in range(max_retries):
                try:
                    website = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((
                            By.XPATH,
                            "//span[contains(@class, 'link-without-visited-state') and @dir='ltr']"
                        ))
                    ).text.strip()
                    data["Website"] = website
                    break
                except TimeoutException:
                    if attempt == max_retries - 1:
                        data["Website"] = "N/A"
                        logger.warning("Website not found after retries")
                    time.sleep(1)

            company_data.append(data)
            logger.info(f"Extracted data for company {index}: {data}")

            # Checkpoint: Save intermediate results
            if index % 5 == 0:
                try:
                    df = pd.DataFrame(company_data)
                    df.to_excel("company_data_intermediate.xlsx", index=False, engine=excel_engine)
                    logger.info(f"Intermediate data saved to company_data_intermediate.xlsx at company {index}")
                except Exception as e:
                    logger.error(f"Error saving intermediate Excel: {e}")

            time.sleep(random.uniform(3, 6))

        except Exception as e:
            logger.error(f"Error processing company {index} ({link}): {e}")
            try:
                with open(f"page_source_company_{index}.html", "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                logger.info(f"Page source saved to page_source_company_{index}.html")
            except Exception as save_error:
                logger.error(f"Failed to save page source: {save_error}")
            company_data.append(data)
            time.sleep(random.uniform(3, 6))
            continue

    # Clean company_data (remove entries with all N/A fields except Link)
    logger.info("Cleaning company_data")
    company_data = [item for item in company_data if any(value != "N/A" for key, value in item.items() if key != "Link")]
    logger.info(f"Cleaned company_data: {len(company_data)} entries")

    # Save final results
    try:
        logger.info("Saving final company data to Excel")
        df = pd.DataFrame(company_data)
        df.to_excel("company_data.xlsx", index=False, columns=["Link", "Name", "Description", "Industry", "Company Size", "Headquarters", "Website"], engine=excel_engine)
        logger.info(f"Final data saved to company_data.xlsx with {len(company_data)} companies")
    except Exception as e:
        logger.error(f"Error saving final Excel: {e}")

except Exception as e:
    logger.error(f"Error in extracting company data: {e}")
    try:
        with open("page_source_step9.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        logger.info("Page source saved to page_source_step9.html")
    except Exception as save_error:
        logger.error(f"Failed to save page source: {save_error}")
    # Save any collected data
    if company_data:
        try:
            df = pd.DataFrame(company_data)
            df.to_excel("company_data_partial.xlsx", index=False, engine=excel_engine)
            logger.info(f"Partial data saved to company_data_partial.xlsx with {len(company_data)} companies")
        except Exception as e:
            logger.error(f"Error saving partial Excel: {e}")
            try:
                df.to_csv("company_data_partial.csv", index=False)
                logger.info("Partial data saved to company_data_partial.csv")
            except Exception as csv_error:
                logger.error(f"Error saving partial CSV: {csv_error}")
    raise Exception("Extraction failed, check logs and page_source_step9.html")

2025-08-01 15:42:32,529 - INFO - Extracting information from each company
2025-08-01 15:42:32,531 - INFO - Processing company 1/30: https://www.linkedin.com/company/cognizant/


2025-08-01 15:42:36,042 - INFO - Company page loaded
2025-08-01 15:42:36,209 - WARNING - Error checking for CAPTCHA/login: Message: invalid selector: Unable to locate an element with the xpath expression //*[しろ*[@id='captcha'] because of the following error:
SyntaxError: Failed to execute 'evaluate' on 'Document': The string '//*[しろ*[@id='captcha']' is not a valid XPath expression.
  (Session info: chrome=138.0.7204.168); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidselectorexception
Stacktrace:
#0 0x60248b86f2ca <unknown>
#1 0x60248b316550 <unknown>
#2 0x60248b31d198 <unknown>
#3 0x60248b31fa28 <unknown>
#4 0x60248b31fab3 <unknown>
#5 0x60248b367b25 <unknown>
#6 0x60248b3682e1 <unknown>
#7 0x60248b3b6621 <unknown>
#8 0x60248b38dbed <unknown>
#9 0x60248b3b39e6 <unknown>
#10 0x60248b38d993 <unknown>
#11 0x60248b359d6b <unknown>
#12 0x60248b35b141 <unknown>
#13 0x60248b8342ab <unknown>
#14 0x60248b8380b9 <un

In [80]:
company_data.clear()
